# Example 4.1.1 / 2: Catching Invalid Data

Companion notebook for **4.1.1 Membership and Cardinality** in the AMPL lab Documentation, section "Catching Invalid Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Catching Invalid Data

If the data puts `wire` in `PRIORITY` but not in `PRODUCTS`, AMPL rejects the data when the set
is first used, before a solver receives a model. This is stronger and more useful than loading
two unrelated sets and hoping they are consistent:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PRODUCTS;
set PRIORITY within PRODUCTS;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PRODUCTS"] = ["bands", "coils", "plate"]
ampl.set["PRIORITY"] = ["bands", "wire"]

In [5]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    print(ampl.get_value("card(PRIORITY)"))
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "display" command:
error processing set PRIORITY:
	PRIORITY contains wire, which is not
	within PRODUCTS;


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils plate;
set PRIORITY := bands wire;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display card(PRIORITY);
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing set PRIORITY:
		PRIORITY contains wire, which is not
		within PRODUCTS;
